# High-Dimensional Feature Spaces

Real ML systems routinely operate in hundreds or thousands of dimensions —
face recognition, text classification, protein structure prediction.  Standard
PCA on such data yields axes driven by whichever features happen to have the
most variance, which often has nothing to do with the classification task.

Sensitivity projection sidesteps this: it measures how much the model's output
changes per unit perturbation along each feature axis, then extracts the three
directions of maximum decision sensitivity.  The resulting projection is always
task-relevant regardless of input dimensionality.

Three high-dimensional real datasets are shown below.

In [ ]:
!pip install -q "geolatent[umap]"

In [ ]:
import numpy as np
import plotly.io as pio
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from geolatent import visualize_decision_geometry, inspect_latent_space, DARK_SCIENTIFIC

pio.renderers.default = "colab"

---
## Olivetti Faces — 4096-D

400 greyscale face images (64×64 pixels = 4096 features), 40 subjects.
We use the first 8 subjects to keep training fast.  Each pixel is a feature;
the sensitivity axes surface the facial regions (eyes, nose, mouth) that
most differentiate subjects under the trained SVM.

In [ ]:
from sklearn.datasets import fetch_olivetti_faces

faces = fetch_olivetti_faces(shuffle=True, random_state=0)
mask = faces.target < 8
X_faces, y_faces = faces.data[mask], faces.target[mask]

print(f"Shape: {X_faces.shape}  |  Classes: {np.unique(y_faces)}")

svm_faces = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=5.0, gamma="scale", probability=True, random_state=0)),
]).fit(X_faces, y_faces)

pixel_names = [f"px_{r}_{c}" for r in range(64) for c in range(64)]

visualize_decision_geometry(
    svm_faces, X_faces, y_faces,
    projection_method="sensitivity",
    feature_names=pixel_names,
    class_names={i: f"Subject {i}" for i in range(8)},
    show_confidence=False,
    show_centroids=True,
    show_ellipsoids=True,
    title="Olivetti Faces — RBF SVM on 4096 pixels (sensitivity)",
).show()

PCA on the same face data picks up the most-variant pixel regions —
typically background and lighting variation rather than identity-discriminative
features.  Compare the axis labels to the sensitivity result above.

In [ ]:
visualize_decision_geometry(
    svm_faces, X_faces, y_faces,
    projection_method="pca",
    feature_names=pixel_names,
    class_names={i: f"Subject {i}" for i in range(8)},
    show_confidence=False,
    show_centroids=True,
    show_ellipsoids=True,
    title="Olivetti Faces — RBF SVM on 4096 pixels (PCA)",
).show()

---
## 20 Newsgroups — TF-IDF Text Features

Classic text classification dataset: newsgroup posts from 4 topics.
TF-IDF vectorisation produces sparse ~10 000-D feature vectors.  A Gradient
Boosting classifier is trained on top; sensitivity projection surfaces the
vocabulary dimensions (words) that most shift the model's category scores.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

categories = ["sci.space", "talk.politics.guns", "rec.sport.hockey", "comp.graphics"]
news = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))

vec = TfidfVectorizer(max_features=300, stop_words="english", min_df=3)
X_news = vec.fit_transform(news.data).toarray()
y_news = news.target

print(f"Shape: {X_news.shape}")

gbm_news = GradientBoostingClassifier(n_estimators=80, max_depth=3, random_state=0).fit(
    X_news, y_news
)

visualize_decision_geometry(
    gbm_news, X_news, y_news,
    projection_method="sensitivity",
    feature_names=vec.get_feature_names_out().tolist(),
    class_names=dict(enumerate(news.target_names)),
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="20 Newsgroups — GBM on 300 TF-IDF features (sensitivity)",
).show()

---
## MNIST Digits — 784-D Pixel Space

Full 28×28 MNIST images as flat pixel vectors.  We use 5000 training samples
across 5 digit classes to keep inference time reasonable.  The latent space
view with t-SNE shows the manifold structure of each digit class; the decision
geometry view with sensitivity shows the pixel regions each classifier splits on.

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
mask = mnist.target.astype(int) < 5
X_mnist = mnist.data[mask][:5000].astype(np.float32) / 255.0
y_mnist = mnist.target[mask][:5000].astype(int)

print(f"Shape: {X_mnist.shape}")

rf_mnist = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0).fit(
    X_mnist, y_mnist
)

px_names = [f"px_{r}_{c}" for r in range(28) for c in range(28)]

visualize_decision_geometry(
    rf_mnist, X_mnist, y_mnist,
    projection_method="sensitivity",
    feature_names=px_names,
    class_names={i: f"Digit {i}" for i in range(5)},
    show_confidence=False,
    show_centroids=True,
    show_ellipsoids=True,
    title="MNIST Digits 0-4 — Random Forest on 784 pixels (sensitivity)",
).show()

Latent space view with t-SNE — reveals the manifold geometry of each digit class
independent of any classifier.

In [ ]:
cfg = DARK_SCIENTIFIC.copy()
cfg.projection.tsne_perplexity = 40.0

inspect_latent_space(
    X_mnist, y_mnist,
    config=cfg.with_method("tsne"),
    show_ellipsoids=True,
    class_names={i: f"Digit {i}" for i in range(5)},
    title="MNIST Digits 0-4 — t-SNE manifold geometry",
).show()